In [ ]:
import sys
import random

def solve():
    # Đọc toàn bộ dữ liệu đầu vào từ stdin
    input_data = sys.stdin.read().split()
    if not input_data:
        return

    N = int(input_data[0])
    K = int(input_data[1])

    # d[i] là thời gian bảo trì tại điểm i (d[0] = 0)
    d = [0] * (N + 1)
    idx = 2
    for i in range(1, N + 1):
        d[i] = int(input_data[idx])
        idx += 1

    # t[i][j] là thời gian di chuyển từ i đến j
    t = []
    for i in range(N + 1):
        row = []
        for j in range(N + 1):
            row.append(int(input_data[idx]))
            idx += 1
        t.append(row)

    # ==========================================
    # CÁC THAM SỐ CỦA THUẬT TOÁN ACO
    # ==========================================
    # LƯU Ý: Với N lên tới 1000, Python có thể bị Time Limit Exceeded (TLE) trên các hệ thống chấm bài.
    # Bạn có thể giảm num_ants và num_iterations xuống để chạy nhanh hơn.
    num_ants = 10         # Số lượng kiến trong 1 thế hệ
    num_iterations = 50   # Số vòng lặp thế hệ
    alpha = 1.0           # Trọng số của Pheromone
    beta = 2.0            # Trọng số của Heuristic (ưu tiên quãng đường ngắn)
    rho = 0.1             # Tốc độ bay hơi của Pheromone
    Q = 10000.0           # Hằng số thả Pheromone

    # Heuristic info (eta) = 1 / (Thời gian di chuyển + thời gian bảo trì)
    eta = [[0.0] * (N + 1) for _ in range(N + 1)]
    for i in range(N + 1):
        for j in range(1, N + 1):
            if i != j:
                cost = t[i][j] + d[j]
                if cost == 0:
                    eta[i][j] = 1e9  # Tránh chia cho 0
                else:
                    eta[i][j] = 1.0 / cost

    # Ma trận Pheromone (tau) khởi tạo bằng 1.0
    tau = [[1.0] * (N + 1) for _ in range(N + 1)]

    best_overall_max_time = float('inf')
    best_overall_routes = []

    # Vòng lặp chính của ACO
    for iteration in range(num_iterations):
        ant_solutions = []

        for ant in range(num_ants):
            routes = [[0] for _ in range(K)]  # Mỗi nhân viên bắt đầu từ 0
            times = [0] * K                   # Thời gian làm việc hiện tại của K nhân viên
            unvisited = set(range(1, N + 1))  # Tập khách hàng chưa được phục vụ

            while unvisited:
                # 1. Tìm nhân viên đang rảnh nhất để cân bằng thời gian (Min-Max)
                min_k = 0
                min_time = float('inf')
                for k in range(K):
                    if times[k] < min_time:
                        min_time = times[k]
                        min_k = k

                curr_node = routes[min_k][-1]

                # 2. Tính xác suất để nhân viên k đi đến các điểm j chưa thăm
                probs = []
                candidates = []
                sum_prob = 0.0

                for j in unvisited:
                    p = (tau[curr_node][j] ** alpha) * (eta[curr_node][j] ** beta)
                    probs.append(p)
                    candidates.append(j)
                    sum_prob += p

                # 3. Chọn điểm tiếp theo (Roulette Wheel Selection)
                if sum_prob == 0:
                    next_node = random.choice(candidates)
                else:
                    r = random.uniform(0, sum_prob)
                    curr_sum = 0.0
                    for idx_prob, p in enumerate(probs):
                        curr_sum += p
                        if curr_sum >= r:
                            next_node = candidates[idx_prob]
                            break
                    else:
                        next_node = candidates[-1]

                # 4. Cập nhật trạng thái
                routes[min_k].append(next_node)
                times[min_k] += t[curr_node][next_node] + d[next_node]
                unvisited.remove(next_node)

            # 5. Tất cả nhân viên quay về trụ sở (0)
            max_time_ant = 0
            for k in range(K):
                last_node = routes[k][-1]
                times[k] += t[last_node][0]
                routes[k].append(0)
                if times[k] > max_time_ant:
                    max_time_ant = times[k]

            ant_solutions.append((max_time_ant, routes))

            # Cập nhật kết quả tốt nhất toàn cục
            if max_time_ant < best_overall_max_time:
                best_overall_max_time = max_time_ant
                best_overall_routes = [list(r) for r in routes]

        # ==========================================
        # CẬP NHẬT PHEROMONE
        # ==========================================
        # 1. Bay hơi Pheromone
        for i in range(N + 1):
            for j in range(N + 1):
                tau[i][j] *= (1.0 - rho)

        # 2. Kiến thả Pheromone (Giải pháp càng tốt, thả càng nhiều)
        for max_time_ant, routes in ant_solutions:
            delta = Q / max_time_ant
            for r in routes:
                for i in range(len(r) - 1):
                    u, v = r[i], r[i+1]
                    tau[u][v] += delta

    # ==========================================
    # IN KẾT QUẢ ĐẦU RA
    # ==========================================
    print(K)
    for k in range(K):
        print(len(best_overall_routes[k]))
        print(" ".join(map(str, best_overall_routes[k])))

if __name__ == '__main__':
    solve()
